In [38]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import pickle
import sys

In [46]:
class CrossAttentionFusionModel(nn.Module):
    def __init__(self, gene_dim=978, chem_dim=1032, hidden_dim=128, num_tokens=32, num_heads=8, dropout=0.2):
        """
        Args:
            gene_dim (int): Number of features in the gene expression data (e.g., 978 for L1000).
            chem_dim (int): Number of features in the chemical descriptors (e.g., 1032 for Morgan + physchem).
            hidden_dim (int): Size of the shared hidden dimension for attention.
            num_heads (int): Number of attention heads.
            dropout (float): Dropout rate to prevent overfitting.
        """
        super().__init__()
        
        # projection layers to map gene and chem features in the same hidden space
        self.hidden_dim = hidden_dim
        self.num_tokens = num_tokens
        self.gene_encoder = nn.Sequential(
            nn.Linear(gene_dim, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(1024, num_tokens * hidden_dim),
            nn.ReLU()
        )

        # Drug Encoder
        self.drug_encoder = nn.Sequential(
            nn.Linear(chem_dim, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(1024, num_tokens * hidden_dim),
            nn.ReLU()
        )

        # Cross Attention
        self.drug_to_gene_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.gene_to_drug_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # LayerNorms
        self.norm_gene = nn.LayerNorm(hidden_dim)
        self.norm_drug = nn.LayerNorm(hidden_dim)

        # Fusion Head
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Regression Head
        self.regressor = nn.Linear(128, 1)

    def forward(self, x_gene, x_drug):

        batch_size = x_gene.size(0)

        # Encode gene features
        # (B,978) -> (B,32*128)
        gene_tokens = self.gene_encoder(x_gene)

        # (B,4096) -> (B,32,128)
        gene_tokens = gene_tokens.view(
            batch_size,
            self.num_tokens,
            self.hidden_dim
        )

        # Encode drug features
        drug_tokens = self.drug_encoder(x_drug)

        drug_tokens = drug_tokens.view(
            batch_size,
            self.num_tokens,
            self.hidden_dim
        )

        # Drug attends to Gene
        drug_attended, drug_gene_weights = (
            self.drug_to_gene_attn(
                query=drug_tokens,
                key=gene_tokens,
                value=gene_tokens
            )
        )

        drug_tokens = self.norm_drug(
            drug_tokens + drug_attended
        )

        gene_attended, gene_drug_weights = (self.gene_to_drug_attn(query=gene_tokens,
                                                                   key=drug_tokens,
                                                                   value=drug_tokens))

        gene_tokens = self.norm_gene(
            gene_tokens + gene_attended
        )

        # Pool tokens
        gene_repr = gene_tokens.mean(dim=1)

        drug_repr = drug_tokens.mean(dim=1)

        # Fusion
        fused = torch.cat(
            [gene_repr, drug_repr],
            dim=1
        )

        fused = self.fusion(fused)
        pred = self.regressor(fused)

        return pred.squeeze(1)

Der DataLoader kümmert sich darum, dass das Modell die Daten in Batches bekommt

In [ ]:
!{sys.executable} -m pip install -q "pyarrow>=13.0.0"

df = pd.read_pickle(
    r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\Datasets\harmonized_data.pkl"
)

In [45]:
# training data
target = 'LN_IC50'
pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
X_chem = df[pharmacophores + list(df.columns[df.columns.str.startswith('Bit_')])].values.astype('float32')
y = df[target].values.astype('float32')

# scale data
gene_scaler = StandardScaler()
chem_scaler = StandardScaler()
X_genomic = gene_scaler.fit_transform(X_genomic)
X_chem = chem_scaler.fit_transform(X_chem)

# split data into train and test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train_val, idx_test = next(gss_test.split(X_genomic, y, groups=df['DRUG_ID']))

# separate validation set from training set
groups_train_val = df['DRUG_ID'].iloc[idx_train_val]
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
idx_train, idx_val = next(gss_val.split(X_genomic[idx_train_val], y[idx_train_val], groups=groups_train_val))

idx_train = idx_train_val[idx_train]
idx_val = idx_train_val[idx_val]

print(X_genomic.shape)
print(X_chem.shape)

(148966, 978)
(148966, 1032)


In [ ]:
class DrugResponseDataset(Dataset):
    def __init__(self, X_gen, X_ch, labels, indices):
        self.X_genomic = torch.tensor(X_gen[indices], dtype=torch.float32)
        self.X_chem = torch.tensor(X_ch[indices], dtype=torch.float32)
        self.y = torch.tensor(labels[indices], dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X_genomic[idx], self.X_chem[idx], self.y[idx]

train_loader = DataLoader(DrugResponseDataset(X_genomic, X_chem, y, idx_train), batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(DrugResponseDataset(X_genomic, X_chem, y, idx_val), batch_size=64, shuffle=False)
test_loader = DataLoader(DrugResponseDataset(X_genomic, X_chem, y, idx_test), batch_size=64, shuffle=False)

In [ ]:
# initialize model, loss function and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CrossAttentionFusionModel(gene_dim=X_genomic.shape[1], chem_dim=X_chem.shape[1], hidden_dim=128, num_tokens=32, num_heads=8, dropout=0.2).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5) # weight decay for regularization, no overfitting
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2) # learning rate scheduler to reduce LR if validation loss plateaus

epochs = 50
best_val_rmse = float("inf")

print("Start Training")

for epoch in range(epochs):
    # training loop
    model.train()
    train_loss = 0.0
    for batch_genes, batch_chem, batch_y in train_loader:
        batch_genes, batch_chem, batch_y = batch_genes.to(device), batch_chem.to(device), batch_y.to(device)
        optimizer.zero_grad()
        predictions = model(batch_genes, batch_chem).unsqueeze(-1) # ensure predictions have shape (Batch, 1) for MSELoss
        predictions = predictions.view(-1) # ensure predictions have the right shape for MSELoss
        batch_y = batch_y.view(-1) # ensure target is the right shape for MSELoss
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_genes.size(0)
    
    total_train_loss = train_loss / len(idx_train)
    
    # validation loop
    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for batch_genes, batch_chem, batch_y in val_loader:
            batch_genes, batch_chem, batch_y = batch_genes.to(device), batch_chem.to(device), batch_y.to(device)
            predictions = model(batch_genes, batch_chem)
            predictions = predictions.view(-1) # ensure predictions have the right shape for MSELoss

            val_preds.extend(predictions.cpu().numpy())
            val_targets.extend(batch_y.cpu().numpy())
    val_preds = np.array(val_preds)
    val_targets = np.array(val_targets)

    val_rmse = root_mean_squared_error(val_targets, val_preds)
    # to save the best model
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        torch.save(model.state_dict(), "best_model.pt")
    val_r2 = r2_score(val_targets, val_preds)

    scheduler.step(val_rmse) # update learning rate based on validation RMSE
    print(f"Epoch {epoch+1:02d}/{epochs} | Train RMSE: {np.sqrt(total_train_loss):.4f} | Val RMSE: {val_rmse:.4f} | Val R²: {val_r2:.4f}")

# final test
model.eval()
test_preds, test_targets = [], []

with torch.no_grad():
    for batch_genes, batch_chem, batch_y in test_loader:
        batch_genes, batch_chem, batch_y = batch_genes.to(device), batch_chem.to(device), batch_y.to(device)
        predictions = model(batch_genes, batch_chem)
        test_preds.extend(predictions.cpu().numpy())
        test_targets.extend(batch_y.cpu().numpy())

# convert lists to numpy arrays for metric calculations
test_preds, test_targets = np.array(test_preds), np.array(test_targets)

# Metrices
test_rmse = root_mean_squared_error(test_targets, test_preds)
test_r2 = r2_score(test_targets, test_preds)
#correlation = np.corrcoef(test_preds, test_targets)[0, 1] # pearson correlation as a simple measure of predictive performance in regression
print(f"Test RMSE: {test_rmse:.4f} | Test R²: {test_r2:.4f} | Test Corr:")

Start Training
